In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoost, Pool

In [4]:
import os
os.listdir('/content/drive/MyDrive/Colab Notebooks/')

['catboost.ipynb', 'train.parquet', 'val.parquet', 'test.parquet']

In [5]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"

df_train = pd.read_parquet(f"{DATA_DIR}/train.parquet")
df_val   = pd.read_parquet(f"{DATA_DIR}/val.parquet")
df_test  = pd.read_parquet(f"{DATA_DIR}/test.parquet")

print(f"train: {df_train.shape}, val: {df_val.shape}, test: {df_test.shape}")

train: (4215569, 65), val: (742778, 65), test: (4959183, 60)


In [6]:
df_train = df_train.sort_values('srch_id').reset_index(drop=True)
df_val   = df_val.sort_values('srch_id').reset_index(drop=True)
df_test  = df_test.sort_values('srch_id').reset_index(drop=True)

In [7]:
exclude_cols = ['srch_id', 'prop_id', 'relevance', 'date_time',
                'click_bool', 'booking_bool', 'position', 'gross_bookings_usd']
features = [c for c in df_train.columns if c not in exclude_cols]

X_train, y_train = df_train[features], df_train['relevance']
X_val,   y_val   = df_val[features],   df_val['relevance']
X_test           = df_test[features]

cat_features = ['site_id', 'prop_country_id', 'srch_destination_id']
cat_features = [c for c in cat_features if c in features]
for c in cat_features:
    for df in [X_train, X_val, X_test]:
        df[c] = df[c].fillna(-1).astype(int).astype(str)

print(f"Features: {len(features)}, categorical: {cat_features}")

/tmp/ipykernel_13585/1260414762.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = df[c].fillna(-1).astype(int).astype(str)
/tmp/ipykernel_13585/1260414762.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = df[c].fillna(-1).astype(int).astype(str)
/tmp/ipykernel_13585/1260414762.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyd

Features: 57, categorical: ['site_id', 'prop_country_id', 'srch_destination_id']


/tmp/ipykernel_13585/1260414762.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = df[c].fillna(-1).astype(int).astype(str)


In [8]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    group_id=df_train['srch_id'],
    cat_features=cat_features,
)
val_pool = Pool(
    data=X_val,
    label=y_val,
    group_id=df_val['srch_id'],
    cat_features=cat_features,
)
test_pool = Pool(
    data=X_test,
    cat_features=cat_features,
)

In [15]:
param_grid = [
    {'depth': 6,  'learning_rate': 0.1,  'l2_leaf_reg': 3},
    {'depth': 8,  'learning_rate': 0.1,  'l2_leaf_reg': 3},
    {'depth': 6,  'learning_rate': 0.05, 'l2_leaf_reg': 3},
    {'depth': 8,  'learning_rate': 0.05, 'l2_leaf_reg': 5},
]

base_params = {
    'loss_function':  'YetiRank',
    'custom_metric':  ['NDCG:top=5'],
    'iterations':     2000,
    'random_seed':    42,
    'task_type':      'GPU',
    'verbose':        50,
    'use_best_model': True,
    'od_type':        'Iter',
    'od_wait':        100,
}

best_params = None
best_ndcg   = -np.inf

for i, hp in enumerate(param_grid):
    params = {**base_params, **hp}
    print(f"\n[{i+1}/{len(param_grid)}] Testing: {hp}")

    model = CatBoost(params)
    model.fit(train_pool, eval_set=val_pool)

    metrics = model.eval_metrics(val_pool, ['NDCG:top=5'])
    metric_key = list(metrics.keys())[0]
    score = max(metrics[metric_key])
    print(f"  NDCG@5 = {score:.4f}  (key: {metric_key})")

    if score > best_ndcg:
        best_ndcg   = score
        best_params = params
        best_model  = model

print(f"\nBest NDCG@5 = {best_ndcg:.4f}")
print(f"Best params: {best_params}")


[1/4] Testing: {'depth': 6, 'learning_rate': 0.1, 'l2_leaf_reg': 3}
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.4159440	best: 0.4159440 (0)	total: 679ms	remaining: 22m 36s
50:	test: 0.4657784	best: 0.4657784 (50)	total: 25.4s	remaining: 16m 10s
100:	test: 0.4750772	best: 0.4750772 (100)	total: 50.1s	remaining: 15m 41s
150:	test: 0.4812594	best: 0.4812594 (150)	total: 1m 14s	remaining: 15m 14s
200:	test: 0.4862927	best: 0.4862927 (200)	total: 1m 39s	remaining: 14m 46s
250:	test: 0.4902649	best: 0.4902649 (250)	total: 2m 3s	remaining: 14m 18s
300:	test: 0.4931414	best: 0.4931414 (300)	total: 2m 27s	remaining: 13m 50s
350:	test: 0.4951408	best: 0.4951408 (350)	total: 2m 51s	remaining: 13m 24s
400:	test: 0.4965979	best: 0.4966184 (399)	total: 3m 15s	remaining: 12m 59s
450:	test: 0.4977222	best: 0.4977624 (449)	total: 3m 38s	remaining: 12m 31s
500:	test: 0.4988078	best: 0.4988078 (500)	total: 4m 2s	remaining: 12m 6s
550:	test: 0.4998189	best: 0.4998189 (550)	total: 4m 26s	remaining: 11m 41s
600:	test: 0.5005420	best: 0.5005420 (600)	total: 4m 50s	remaining: 11m 15s
650:	test: 0.5012986	bes

Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.4189097	best: 0.4189097 (0)	total: 780ms	remaining: 25m 59s
50:	test: 0.4711176	best: 0.4711176 (50)	total: 32.7s	remaining: 20m 48s
100:	test: 0.4810515	best: 0.4810515 (100)	total: 1m 3s	remaining: 19m 53s
150:	test: 0.4861393	best: 0.4861393 (150)	total: 1m 33s	remaining: 19m 5s
200:	test: 0.4903274	best: 0.4903274 (200)	total: 2m 3s	remaining: 18m 24s
250:	test: 0.4932563	best: 0.4932563 (250)	total: 2m 32s	remaining: 17m 45s
300:	test: 0.4959991	best: 0.4960131 (299)	total: 3m 2s	remaining: 17m 9s
350:	test: 0.4980687	best: 0.4980855 (349)	total: 3m 31s	remaining: 16m 34s
400:	test: 0.4995950	best: 0.4995950 (400)	total: 4m	remaining: 15m 57s
450:	test: 0.5003925	best: 0.5004068 (449)	total: 4m 29s	remaining: 15m 23s
500:	test: 0.5013491	best: 0.5014798 (497)	total: 4m 57s	remaining: 14m 50s
550:	test: 0.5022732	best: 0.5022732 (550)	total: 5m 26s	remaining: 14m 18s
600:	test: 0.5027077	best: 0.5027230 (587)	total: 5m 54s	remaining: 13m 45s
650:	test: 0.5034931	best: 0.

Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.4159440	best: 0.4159440 (0)	total: 604ms	remaining: 20m 6s
50:	test: 0.4572485	best: 0.4572485 (50)	total: 25.8s	remaining: 16m 27s
100:	test: 0.4657076	best: 0.4657076 (100)	total: 51.2s	remaining: 16m 2s
150:	test: 0.4710621	best: 0.4710621 (150)	total: 1m 16s	remaining: 15m 32s
200:	test: 0.4754267	best: 0.4754267 (200)	total: 1m 40s	remaining: 15m 1s
250:	test: 0.4782782	best: 0.4783541 (247)	total: 2m 5s	remaining: 14m 31s
300:	test: 0.4810459	best: 0.4810459 (300)	total: 2m 29s	remaining: 14m 4s
350:	test: 0.4836588	best: 0.4836588 (350)	total: 2m 53s	remaining: 13m 37s
400:	test: 0.4859721	best: 0.4859721 (400)	total: 3m 18s	remaining: 13m 10s
450:	test: 0.4880101	best: 0.4880685 (449)	total: 3m 42s	remaining: 12m 43s
500:	test: 0.4899919	best: 0.4899919 (500)	total: 4m 6s	remaining: 12m 17s
550:	test: 0.4913689	best: 0.4913689 (550)	total: 4m 30s	remaining: 11m 51s
600:	test: 0.4927555	best: 0.4927588 (595)	total: 4m 54s	remaining: 11m 25s
650:	test: 0.4938720	best: 

Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.4189763	best: 0.4189763 (0)	total: 733ms	remaining: 24m 24s
50:	test: 0.4648397	best: 0.4648397 (50)	total: 32.6s	remaining: 20m 44s
100:	test: 0.4703703	best: 0.4703703 (100)	total: 1m 4s	remaining: 20m 15s
150:	test: 0.4768117	best: 0.4768117 (150)	total: 1m 35s	remaining: 19m 28s
200:	test: 0.4809308	best: 0.4809540 (198)	total: 2m 6s	remaining: 18m 49s
250:	test: 0.4833977	best: 0.4834602 (248)	total: 2m 36s	remaining: 18m 11s
300:	test: 0.4861748	best: 0.4861748 (300)	total: 3m 7s	remaining: 17m 36s
350:	test: 0.4882496	best: 0.4882496 (350)	total: 3m 37s	remaining: 17m
400:	test: 0.4907734	best: 0.4907734 (400)	total: 4m 6s	remaining: 16m 24s
450:	test: 0.4921246	best: 0.4921323 (445)	total: 4m 36s	remaining: 15m 50s
500:	test: 0.4935887	best: 0.4936184 (499)	total: 5m 6s	remaining: 15m 15s
550:	test: 0.4949063	best: 0.4949063 (550)	total: 5m 35s	remaining: 14m 42s
600:	test: 0.4963249	best: 0.4963411 (599)	total: 6m 4s	remaining: 14m 8s
650:	test: 0.4970837	best: 0.49

In [16]:
df_all = pd.concat([df_train, df_val], ignore_index=True).sort_values('srch_id').reset_index(drop=True)
X_all  = df_all[features]
y_all  = df_all['relevance']
for c in cat_features:
    X_all[c] = X_all[c].fillna(-1).astype(int).astype(str)

all_pool = Pool(
    data=X_all,
    label=y_all,
    group_id=df_all['srch_id'],
    cat_features=cat_features,
)

final_params = {**best_params, 'iterations': best_model.best_iteration_, 'use_best_model': False, 'od_wait': None}
final_model  = CatBoost(final_params)
final_model.fit(all_pool)
print("Final model trained.")

/tmp/ipykernel_13585/2123820030.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_all[c] = X_all[c].fillna(-1).astype(int).astype(str)
/tmp/ipykernel_13585/2123820030.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_all[c] = X_all[c].fillna(-1).astype(int).astype(str)
/tmp/ipykernel_13585/2123820030.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://p

Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 785ms	remaining: 17m 18s
50:	total: 34.8s	remaining: 14m 29s
100:	total: 1m 7s	remaining: 13m 36s
150:	total: 1m 39s	remaining: 12m 52s
200:	total: 2m 11s	remaining: 12m 12s
250:	total: 2m 42s	remaining: 11m 33s
300:	total: 3m 12s	remaining: 10m 55s
350:	total: 3m 43s	remaining: 10m 18s
400:	total: 4m 13s	remaining: 9m 43s
450:	total: 4m 43s	remaining: 9m 8s
500:	total: 5m 13s	remaining: 8m 35s
550:	total: 5m 43s	remaining: 8m 1s
600:	total: 6m 12s	remaining: 7m 28s
650:	total: 6m 42s	remaining: 6m 55s
700:	total: 7m 11s	remaining: 6m 23s
750:	total: 7m 41s	remaining: 5m 51s
800:	total: 8m 10s	remaining: 5m 20s
850:	total: 8m 40s	remaining: 4m 49s
900:	total: 9m 9s	remaining: 4m 18s
950:	total: 9m 38s	remaining: 3m 47s
1000:	total: 10m 8s	remaining: 3m 16s
1050:	total: 10m 37s	remaining: 2m 45s
1100:	total: 11m 6s	remaining: 2m 15s
1150:	total: 11m 35s	remaining: 1m 44s
1200:	total: 12m 4s	remaining: 1m 14s
1250:	total: 12m 33s	remaining: 44s
1300:	total: 13m 2s	remaining: 13

In [18]:
df_test['predicted_score'] = final_model.predict(test_pool)

submission = (
    df_test
    .sort_values(['srch_id', 'predicted_score'], ascending=[True, False])
    [['srch_id', 'prop_id']]
    .rename(columns={'srch_id': 'SearchId', 'prop_id': 'PropertyId'})
)

submission.to_csv(f"{DATA_DIR}/submission_catboost.csv", index=False)
print(f"Submission saved: {len(submission)} rows")
submission.head(10)

Submission saved: 4959183 rows


,SearchId,PropertyId
23,1,99484
13,1,28181
9,1,54937
5,1,61934
3,1,50162
0,1,5543
12,1,34263
28,1,3180
27,1,90385
10,1,24194
